In [1]:
suppressPackageStartupMessages({
    library(jsonlite)
    library(tidyverse)
})

# split lineages into chunks 

In [3]:
cohort = 'EDP1-EDP2-ARB'
xen_basepath = file.path('/data/srlab/AMP_collab/data/early_disease_synovium/xenium/combined/', cohort, 'lineages')
params = data.frame(
    lineage = c('B_plasma', 'Endothelial', 'Myeloid', 'Stromal', 'T_NK'), 
    cohort = rep(cohort, times = 5), 
    max_mult = rep(1.25, times = 5)
    )

lines <- lapply(seq_len(nrow(params)), function(i) {
    toJSON(as.list(params[i, ]), auto_unbox = TRUE)
})
outpath = file.path(
    "/data/srlab/AMP_collab/lakshay-yakir/5.coarse_types/", 
    paste0("1.split_lineages_into_chunks_", cohort, "_params.jsonl")
    )
writeLines(unlist(lines), outpath)

# lineages to cell types 

In [13]:
cohort = 'EDP1-EDP2-ARB'
xen_basepath = file.path('/data/srlab/AMP_collab/lakshay-yakir/4.lineages/out_rds/', cohort, 'lineages')

params = data.frame()
for (lineage in c('B_plasma', 'Endothelial', 'Myeloid', 'Stromal', 'T_NK')) {
    chunks_path = file.path(xen_basepath, paste0(lineage, '_chunks'))
    chunks = list.files(chunks_path, full.names = TRUE)
    lineage_params = data.frame(
        lineage = lineage, 
        xen_path = chunks, 
        cohort = cohort, 
        batch_vars = 'cohort'
        )
    params <- dplyr::bind_rows(params, lineage_params)
}

lines <- lapply(seq_len(nrow(params)), function(i) {
    toJSON(as.list(params[i, ]), auto_unbox = TRUE)
})
outpath = file.path(
    "/data/srlab/AMP_collab/lakshay-yakir/5.coarse_types/", 
    paste0("2.lineages_to_celltypes_bplasmaonly_", cohort, "_params.jsonl")
    )
writeLines(unlist(lines), outpath)

# combine all chunks and generate umap for each coarse type

NB: CCA weights are used for T and plasma cell UMAPs. These UMAPs are generated for potential downstream use but are not directly used by the fine typing code. 

In [14]:
cohort = 'EDP1-EDP2-ARB'
xen_basepath = file.path('/data/srlab/AMP_collab/lakshay-yakir/5.coarse_types/out_rds/', cohort)

params = data.frame()
for (lineage in c('B_plasma', 'Endothelial', 'Myeloid', 'Stromal', 'T_NK')) {
    cts <- list.dirs(file.path(xen_basepath, lineage), full.names = TRUE, recursive = FALSE)
    cts <- cts[basename(cts) != 'Untyped']
    lineage_params = data.frame(
        ct = basename(cts),
        lineage = lineage,
        cohort = cohort, 
        cca_path = ifelse(basename(cts) %in% c('T', 'Plasma'), # removing B for now as gene expression correlations looked bad
                              paste0('/data/srlab/AMP_collab/lakshay-yakir/5.coarse_types/out_ccaweights/', basename(cts), '_ccaweights.RDS'), 
                              ""),
        batch_vars = 'cohort'
        )
    params <- dplyr::bind_rows(params, lineage_params)     
}

lines <- lapply(seq_len(nrow(params)), function(i) {
    toJSON(as.list(params[i, ]), auto_unbox = TRUE)
})
outpath = file.path(
    "/data/srlab/AMP_collab/lakshay-yakir/5.coarse_types/", 
    paste0("3.combine_coarsetype_chunks_and_umap_", cohort, "_params.jsonl")
    )
writeLines(unlist(lines), outpath)